# Pilot Test: Jupyter + Public LLM Memory Dump Feasibility

이 노트북은 사용자가 지정한 memory dump를 읽고, 로컬 분석 결과를 공개 LLM API에 연결하는 최소 동작 데모입니다.

In [1]:
from pathlib import Path
import json
import os
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DUMP_PATH = os.environ.get('DUMP_PATH', '').strip()
VMLINUX_PATH = os.environ.get('VMLINUX_PATH', '').strip()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DUMP_PATH =', DUMP_PATH)
print('VMLINUX_PATH =', VMLINUX_PATH or '(not provided)')
print('dump exists =', Path(DUMP_PATH).exists() if DUMP_PATH else False)
print('OPENAI_API_KEY configured =', bool(os.environ.get('OPENAI_API_KEY')))

PROJECT_ROOT = /home/taejin/Jupyter/jupyter-ramdump-analyzer
DUMP_PATH = /home/taejin/Jupyter/data/memory/memory.vmem
VMLINUX_PATH = (not provided)
dump exists = True
OPENAI_API_KEY configured = True


## Imports and runtime options

In [2]:
from analysis_pipeline import FeasibilityOptions, run_feasibility_analysis
from llm_assistant import LLMAssistant
from memory_kernel_analyzer import build_analysis_context, summarize_findings

MODEL = os.environ.get('OPENAI_MODEL', 'openai/gpt-oss-120b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')
ALLOW_RAW_CHUNK_EXCERPT = True
GENERATE_NEXT_STEPS = False

assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('MODEL =', MODEL)
print('API_BASE =', API_BASE)
assistant.is_configured()

MODEL = openai/gpt-oss-120b:free
API_BASE = https://openrouter.ai/api/v1


True

## Step 1. Build local analysis context

In [3]:
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 2. Inspect chunk samples sent to the LLM

In [4]:
for sample in context.raw_chunk_samples[:3]:
    print(sample)
    print('-' * 80)

{'offset': 0, 'size': 256, 'hex_preview': '53ff00f053ff00f0c3e200f053ff00f053ff00f054ff00f0888400f053ff00f0', 'ascii_preview': 'S...S.......S...S...T.......S...........V...V...V...V...W...........M...A.......9...Y...."J.....Y.......n...S...S.......'}
--------------------------------------------------------------------------------
{'offset': 536870912, 'size': 256, 'hex_preview': '280e3f001b0129001f0530001b0129001f0530001b0129001f0530001f053000', 'ascii_preview': '(.?...)...0...)...0...)...0...0...)...0...0...)...0...)...0...)...0...)...0...)...0...0...0...0...)...0...)...0...0...).'}
--------------------------------------------------------------------------------
{'offset': 1073741824, 'size': 256, 'hex_preview': 'fe86fe2124ed8f0525bda90b25ce981122bd0b1425fd8d1a252d872025ce8526', 'ascii_preview': '...!$...%...%..."...%...%-. %..&"(....)%../%|."..8%..>%..D%l.".x"..P%=.V%].\\%].b%..h%}.n%M.t%..z%...%.."...%...%.."...L.'}
-------------------------------------------------------------------

## Step 3. Run the feasibility pipeline

In [5]:
options = FeasibilityOptions(
    dump_path=DUMP_PATH,
    allow_raw_chunk_excerpt=ALLOW_RAW_CHUNK_EXCERPT,
    raw_chunk_limit=2,
    generate_next_steps=GENERATE_NEXT_STEPS,
)
report = run_feasibility_analysis(DUMP_PATH, assistant, options)
report.llm_enabled

True

## Step 4. Review local findings

In [6]:
print(json.dumps(report.findings, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 5. Review LLM output

API 키가 없으면 이 셀은 `None` 또는 오류 메시지를 보여줍니다.

In [7]:
print(report.llm_analysis)
print('\n' + '=' * 80 + '\n')
print(report.next_steps)

### 1️⃣ 핵심 이상 징후 (Observed Anomalies)

| 구분 | 내용 | 근거 |
|------|------|------|
| **다중 커널 패닉** | `kernel_oops:BUG:` 와 `crashes:crash` 패턴이 각각 2회 이상 발견 | `panic_pattern_count: 2` |
| **메모리 할당 오류 문자열** | `alloc magic is broken at %p: %lx`, `out of memory` | 메모리 관리 서브시스템에서 할당 검증 실패 혹은 OOM 상황을 나타냄 |
| **GRUB 메모리 함수 호출 흔적** | `grub_calloc`, `grub_malloc`, `grub_realloc`, `grub_zalloc` | 부팅 단계에서 메모리 할당이 비정상적으로 진행됐을 가능성 |
| **PXE 복사 오류** | `PXE‑E20: BIOS extended memory copy error.` | 네트워크 부팅 시 BIOS‑레벨 메모리 복사 실패, 물리 메모리 손상 혹은 DMA 오류와 연관될 수 있음 |
| **대용량 메모리 덤프(4 GB) 내 비정상 패턴** | 샘플 0/0x20000000(≈512 MiB)에서 `53ff00f0…` 같은 반복적인 0xF0 패턴이 다수 발견 | 특정 영역이 **패딩/클리어링** 되었거나, 하드웨어 오류(비트 플립)로 인해 동일 바이트가 반복되는 현상 |
| **프로세스·네트워크 비정상** | 외부 IP 17개, 다수 포트(FTP, SSH, Telnet, SMTP, DNS, HTTPS) 열림 | 시스템이 **외부와 과도하게 통신**하고 있거나, 침해 시도·악성 코드 활동이 있었을 가능성 |
| **단일 사용자 계정** (`woreilly`) | 프로세스 50개가 모두 동일 사용자 아래 실행 | 권한 상승/루트 권한 획득 후 하나의 계정으로 작업을 수행했을 가능성 |
| **심볼 파일 부재** | `vmlinux` 없이 dump‑only 분석 | 정확한 스택/레지스터 해석이 제한

## Step 6. Script generation example

In [8]:
if assistant.is_configured():
    script = assistant.generate_analysis_script(
        'memory dump에서 panic signal과 call trace 후보를 추출하는 Python 스크립트',
        findings=report.findings,
    )
    print(script)
else:
    print('OPENAI_API_KEY is not configured')

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
memory_panic_extractor.py
- 주어진 .vmem 메모리 덤프 파일에서 커널 패닉 신호와
  call‑trace(함수 이름/주소) 후보를 추출한다.
- vmlinux 심볼이 없으므로 문자열 기반 탐색에 의존한다.
- 큰 파일을 전체 로드하지 않고 chunk 단위로 순차 읽는다.
"""

import argparse
import re
import sys
from pathlib import Path

# ----------------------------------------------------------------------
# 패턴 정의
# ----------------------------------------------------------------------
# 커널 패닉/오프스 문자열 (예시)
PANIC_PATTERNS = [
    rb'kernel_oops:BUG:',
    rb'kernel panic',
    rb'BUG:',
    rb'panic',
    rb'crash',
]

# call‑trace 라인에 흔히 보이는 포맷:
#   [<addr>] function_name+0xoffset/0xsize
CALL_TRACE_RE = re.compile(
    rb'\[<([0-9a-fA-F]+)>\]\s+([^\s+]+)(?:\+0x[0-9a-fA-F]+)?'
)

# ----------------------------------------------------------------------
# 파일을 chunk 로 읽으며 매칭을 수행하는 제네레이터
# ----------------------------------------------------------------------
def iter_chunks(fp, chunk_size=4 * 1024 * 1024):
    """파일을 지정된 크기의 청크로 순